# M1.3 · Tools (herramientas) 🧮

**Lección oficial:** *Tools* · **Notebook original:** `module-1/1.2_tools.ipynb`

## 🎯 Qué vas a aprender
- Qué es una **tool** (herramienta) y cómo se crea con `@tool`.
- Por qué la **docstring** es tan importante.
- Cómo un agente **decide** usar una tool y cómo ver esa decisión (`tool_calls`).

## 🧸 Explicado sencillo
Un modelo es malo haciendo cuentas exactas: adivina. Una **tool** es como darle una **calculadora** 🧮.

El modelo no aprieta los botones: **pide** "usa la calculadora con 467", el agente la usa por él y le devuelve el resultado.

## 🛠️ Paso 0 · Preparar el notebook

Esta celda hace tres cosas por ti:
1. Encuentra la carpeta principal del repo (donde vive `local_model.py`).
2. Lee tu archivo `.env` (tus llaves y la dirección del servidor) **sin mostrarlas**.
3. Revisa el **semáforo** del servidor Ollama, que es **compartido**:
   - 🟢 libre → adelante.
   - 🟡 hay un modelo cargado → si fuiste tú hace un rato, sigue; si no, espera.
   - 🔴 sin conexión → revisa `OLLAMA_BASE_URL` en tu `.env`.

> Corre las celdas **en orden, de arriba hacia abajo** (Shift + Enter). El número `[n]` a la izquierda te dice en qué orden se ejecutaron.

In [ ]:
# 🟢 ADAPTADO LOCAL: preparar entorno (ver ayuda.py)
from ayuda import preparar, ver, semaforo
preparar()

from local_model import get_model   # se importa DESPUÉS de cargar el .env

## 1 · Crear una tool con `@tool`
Una tool es **una función normal de Python** con el decorador `@tool` encima.

El modelo **no ve tu código**. Solo ve tres cosas:

🧠 **Mnemotecnia N-D-A:**
- **N**ombre → `square_root`.
- **D**ocstring → el texto entre `"""…"""` (qué hace).
- **A**rgumentos → `x: float` (qué necesita).

Si la docstring es confusa, el modelo no sabrá cuándo usarla.

In [ ]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

### Lo que el modelo "ve" de tu tool

In [ ]:
print("Nombre:     ", square_root.name)
print("Descripción:", square_root.description)
print("Argumentos: ", square_root.args)

### Otras formas de escribir la misma tool
Puedes ponerle otro nombre a la función, o dar la descripción por fuera. El resultado para el modelo es igual.

In [ ]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

print(tool1.name, "→", tool1.description)

## 2 · Probar la tool a mano
Antes de dársela al agente, la probamos nosotros. Ojo: se le pasa un **diccionario** con los argumentos.

In [ ]:
square_root.invoke({"x": 467})

## 3 · Darle la tool al agente 🧮
Ahora el agente tiene una calculadora. Mira cómo la usa.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre
# agent = create_agent(model="gpt-5-nano", tools=[tool1], system_prompt=...)

# 🟢 ADAPTADO LOCAL
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model=get_model(),
    tools=[square_root],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root of any number.",
)

response = agent.invoke({"messages": [HumanMessage(content="What is the square root of 467?")]})
print(response["messages"][-1].content)

## 4 · Mirar por dentro: el bucle del agente 🔁
Con `ver()` vas a ver **4 mensajes**. Ese es el famoso **bucle del agente**:

1. 🧑 Humano pregunta.
2. 🤖 Modelo **pide** usar la tool (todavía no responde).
3. 🔧 La tool devuelve el resultado.
4. 🤖 Modelo usa el resultado y responde.

🧠 **Mnemotecnia P-P-R-R:** *Pregunta → Pide tool → Resultado → Responde.*

In [ ]:
ver(response)

In [ ]:
# La "petición" del modelo está en tool_calls del mensaje [1]
print(response["messages"][1].tool_calls)

🤔 **Pregunta para pensar:** si preguntas *"¿Cuál es la capital de Francia?"*, ¿crees que el agente usará la calculadora? Pruébalo en la celda de abajo.

<details><summary>Ver respuesta</summary>

No debería: el modelo lee la docstring ("raíz cuadrada") y decide que no le sirve. **El modelo decide** cuándo usar una tool.
</details>

In [ ]:
response = agent.invoke({"messages": [HumanMessage(content="What is the capital of France?")]})
ver(response)

## ✍️ Tu turno
1. Crea una tool `square(x: float)` que devuelva `x ** 2`, con una buena docstring.
2. Dale **las dos tools** al agente: `tools=[square_root, square]`.
3. Pregúntale: *"What is the square of 12 and the square root of 144?"* y mira con `ver()` si usa las dos.
4. Reto: escribe una docstring **mala** (por ejemplo `"""hace cosas"""`) y mira si el agente se confunde.

## ✅ Mini quiz
1. ¿Qué tres cosas de una tool ve el modelo?
2. ¿Quién **ejecuta** la tool: el modelo o el agente?
3. ¿En qué parte del mensaje se ve que el modelo pidió una tool?
4. ¿Cuántos mensajes hay en una respuesta que usó una tool una vez?

<details><summary>Respuestas</summary>

1. Nombre, docstring (descripción) y argumentos (N-D-A).
2. El agente. El modelo solo la pide.
3. En `tool_calls` del `AIMessage`.
4. Cuatro: Humano, Modelo pide, Tool, Modelo responde.
</details>

## 🧠 Resumen en una línea
**Tool = calculadora del modelo. El modelo la PIDE, el agente la USA. Lo que importa: Nombre, Docstring y Argumentos.**